# Phase 1 — Traditional Computer Vision Star Detection Demonstration

This lightweight notebook demonstrates the end-to-end Phase 1 traditional star detection pipeline using the modular `constellation_detection` package:

1. **Global Threshold** (`global_threshold`)
2. **Adaptive Top-Hat** (`CV2_AdaptiveTopHat`)
3. **Multi-Scale DoG** (`CV3_MultiScaleDoG`)

### Dataset Provenance (Optional ESA/Hubble Samples)
- **Cassiopeia**: https://cdn.esahubble.org/archives/images/large/opo0613c.jpg
- **Ursa Major**: https://cdn.esahubble.org/archives/images/large/opo0706b.jpg
- **Ursa Minor**: https://cdn.esahubble.org/archives/images/large/heic0401g.jpg
- **Orion**: https://cdn.esahubble.org/archives/images/large/opo0205b.jpg
- **Cygnus**: https://cdn.esahubble.org/archives/images/large/heic0720c.jpg

> **Note**: Detector CSV outputs are unverified predictions stored under `outputs/predictions/` and are kept strictly separate from verified ground-truth annotations. Precision, recall, and F1 are intentionally omitted until verified ground truth is introduced.

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

from constellation_detection.config import ProjectPaths
from constellation_detection.evaluation import (
    build_detector_leaderboard,
    evaluate_image_detectors,
)
from constellation_detection.image import load_image
from constellation_detection.visualization import plot_detector_diagnostics

# Configurable paths (override via CONSTELLATION_DATA_DIR / CONSTELLATION_OUTPUT_DIR)
paths = ProjectPaths.from_paths()
RAW_DATA_DIR = paths.data_dir / "raw"
PREDICTIONS_DIR = paths.output_dir / "predictions"
VISUALIZATIONS_DIR = paths.output_dir / "visualizations"

# Experiment configuration
MAX_DIM = 800
MIN_DIST_PX = 10.0
MAX_CANDIDATES = 250
TOP_K_SNR = 30
SYNTHETIC_SEED = 42

print("Project root:", paths.project_root)
print("Raw data dir:", RAW_DATA_DIR)
print("Predictions dir:", PREDICTIONS_DIR)
print("Visualizations dir:", VISUALIZATIONS_DIR)

In [ ]:
# Load local raw astronomical images if available; otherwise generate a seeded synthetic starfield
image_paths = sorted(
    p for p in RAW_DATA_DIR.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
) if RAW_DATA_DIR.exists() else []

loaded_images: list[tuple[str, np.ndarray]] = []
if image_paths:
    for img_path in image_paths:
        loaded_images.append((img_path.name, load_image(img_path)))
    print(f"Loaded {len(loaded_images)} astronomical image(s) from {RAW_DATA_DIR}")
else:
    print(
        "No local images found in data/raw/. "
        "Run 'python scripts/download_phase1_samples.py' to fetch optional ESA/Hubble samples.\n"
        "Falling back to a deterministic synthetic starfield for demonstration."
    )
    rng = np.random.default_rng(SYNTHETIC_SEED)
    h, w = 512, 640
    yy, xx = np.mgrid[0:h, 0:w]
    bg = (12.0 + 18.0 * (xx / w) + rng.normal(0.0, 2.0, size=(h, w))).clip(0, 255)
    synth = bg.astype(np.uint8)
    for _ in range(35):
        cx = int(rng.integers(20, w - 20))
        cy = int(rng.integers(20, h - 20))
        radius = int(rng.integers(2, 4))
        intensity = int(rng.integers(180, 256))
        cv2.circle(synth, (cx, cy), radius, intensity, -1)
    loaded_images.append(("synthetic_starfield_demo.png", cv2.cvtColor(synth, cv2.COLOR_GRAY2BGR)))

In [ ]:
# Run all three detectors, export prediction CSVs, and render 4-panel diagnostic PNGs
all_metrics: list[pd.DataFrame] = []

for image_name, image_bgr in loaded_images:
    metrics_df, results, working_img = evaluate_image_detectors(
        image=image_bgr,
        image_name=image_name,
        predictions_dir=PREDICTIONS_DIR,
        max_dim=MAX_DIM,
        min_dist_px=MIN_DIST_PX,
        max_candidates=MAX_CANDIDATES,
        top_k_snr=TOP_K_SNR,
    )
    all_metrics.append(metrics_df)

    stem = Path(image_name).stem
    for res in results:
        detector_name = res["detector"]
        fig, saved_png = plot_detector_diagnostics(
            image_bgr=working_img,
            result=res,
            output_dir=VISUALIZATIONS_DIR,
            filename=f"{stem}_{detector_name}_diagnostics.png",
            max_dim=MAX_DIM,
        )
        display(fig)
        print(f"Saved diagnostic panel: {saved_png}")

combined_metrics = pd.concat(all_metrics, ignore_index=True)
leaderboard = build_detector_leaderboard(combined_metrics)

print("\n=== Per-Image Detector Metrics ===")
display(combined_metrics)

print("\n=== Aggregated Detector Leaderboard ===")
display(leaderboard)